In [ ]:
import torch
import torch.nn as nn
import numpy as np

In [ ]:
torch.manual_seed(1)

Build transformer and classification head

In [ ]:
# transformer encoders
# input as embedding_dim=(Batch, token count, embedding size)
#----------------------------------
# input
# normalize (layer norm)
# MHA
# add input (residual connection)
# input2
# normalize (layer norm)
# MLP
# add input2 (residual connection)
#----------------------------------
class transformer_block(nn.Module):
    def __init__(self, embedding_dim, num_heads, dropout):
        super().__init__()
        self.layer_norm1 = nn.LayerNorm(embedding_dim)
        self.mha = nn.MultiheadAttention(embedding_dim,
                                         num_heads,
                                         dropout,
                                         batch_first=True)
        #self.dropout = nn.Dropout(dropout)

        self.layer_norm2 = nn.LayerNorm(embedding_dim)
        hidden_dim=int(embedding_dim*1.5)
        self.mlp = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, embedding_dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        x1 = self.layer_norm1(x) # layer norm
        # x2, _ = self.mha(x1, x1, x1) # multi head self attention
        x2, _ = self.mha(x1, x1, x1, need_weights=False)
        x2 = x2 + x # residual connection

        x3 = self.layer_norm2(x2) # layer norm
        x3 = self.mlp(x3) # fc layer
        x3 = x3 + x2
        return x3

# multiple transformer blocks
class transformer_encoder(nn.Module):
    def __init__(self, num_blocks, embedding_dim, num_heads, dropout=0):
        super().__init__()
        self.blocks = nn.ModuleList([
            transformer_block(embedding_dim, num_heads, dropout) for _ in range(num_blocks)
        ])

    def forward(self, x):
        for block in self.blocks:
            x = block(x)
        return x

#--------------------------------
# use first one as input for MLP
# predict
#--------------------------------
class classification_head(nn.Module):
    def __init__(self, embedding_dim, num_class=10, dropout=0):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(embedding_dim, embedding_dim//2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embedding_dim//2, num_class),
        )
    def forward(self, x):
        return self.mlp(x)

class vision_transformer(nn.Module):
    def __init__(self, encoder, head):
        super().__init__()
        self.encoder = encoder
        self.head = head

    def forward(self, x):
        x = self.encoder(x)
        class_token = x[:, 0, :] # take first token for classification
        x = self.head(class_token)
        return x

Test input, data

outputs logit

In [ ]:
# input dim (N+1, D)
b, n, m = 2, 3, 4

# input
input = torch.normal(mean=0, std=1, size=(b, n, m))

num_heads = 2
dropout=0

# model inifialization and forward pass
encoder = transformer_encoder(8, m, num_heads, dropout)
head = classification_head(m, 10, dropout)

vit = vision_transformer(encoder, head)
output = vit(input)
output

tensor([[-0.4710, -0.0205,  0.2214, -0.5123,  0.3720,  0.1179, -0.4594,  0.4689,
         -0.7284, -0.5056],
        [-0.6119, -0.3126,  0.4808, -0.8487,  0.2859, -0.0228, -0.7189,  0.0967,
         -0.6446, -0.2013]], grad_fn=<AddmmBackward0>)